# Amazon Software Dataset - Deep Exploratory Data Analysis

This notebook performs comprehensive EDA on the Amazon Reviews 2023 dataset (Software category) to understand:
- Review dataset characteristics and patterns
- Metadata dataset characteristics and patterns
- Relationships between reviews and products
- Data quality issues and insights

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

## 1. Load Datasets

In [ ]:
import os
from datasets import load_dataset
from tqdm import tqdm

# Configuration
data_dir = "./output"
review_file = f"{data_dir}/review_data.parquet"
meta_file = f"{data_dir}/meta_data.parquet"

# Check if parquet files exist
if os.path.exists(review_file) and os.path.exists(meta_file):
    print("Loading datasets from parquet files...")
    review_df = pd.read_parquet(review_file)
    meta_df = pd.read_parquet(meta_file)
    print(f"\nReview dataset loaded: {review_df.shape[0]:,} rows, {review_df.shape[1]} columns")
    print(f"Metadata dataset loaded: {meta_df.shape[0]:,} rows, {meta_df.shape[1]} columns")
else:
    print("Parquet files not found!")
    print(f"Expected files at:")
    print(f"  - {review_file}")
    print(f"  - {meta_file}")
    print("\nPlease run 'extract_amazon_dataset.ipynb' first to download and export the data.")
    print("\nAlternatively, you can load directly from HuggingFace (this will take ~10 minutes):")
    print("Uncomment and run the code block below to load from HuggingFace.\n")

    # OPTION: Uncomment this block to load directly from HuggingFace
    """
    dataset_name = "McAuley-Lab/Amazon-Reviews-2023"
    category = "Software"

    print("Loading review dataset from HuggingFace...")
    review_dataset = load_dataset(dataset_name, f"raw_review_{category}", trust_remote_code=True)
    all_reviews = [review for review in tqdm(review_dataset["full"])]
    review_df = pd.DataFrame(all_reviews)

    print("Loading metadata dataset from HuggingFace...")
    meta_dataset = load_dataset(dataset_name, f"raw_meta_{category}", split="full", trust_remote_code=True)
    meta_df = pd.DataFrame(meta_dataset)

    # Optional: Save to parquet for future use
    os.makedirs(data_dir, exist_ok=True)
    review_df.to_parquet(review_file, index=False)
    meta_df.to_parquet(meta_file, index=False)
    print(f"\nDatasets saved to {data_dir}/")
    """

    raise FileNotFoundError(f"Required parquet files not found. Please run 'extract_amazon_dataset.ipynb' first.")

## 2. Dataset Overview

### 2.1 Review Dataset Overview

In [ ]:
print("=" * 80)
print("REVIEW DATASET OVERVIEW")
print("=" * 80)

print("\nDataset Shape:")
print(f"  Rows: {review_df.shape[0]:,}")
print(f"  Columns: {review_df.shape[1]}")

print("\nColumn Information:")
review_df.info()

In [ ]:
print("\nFirst few rows:")
review_df.head()

In [ ]:
print("\nBasic Statistics:")
review_df.describe()

In [ ]:
# Memory usage
print("\nMemory Usage:")
memory_usage = review_df.memory_usage(deep=True) / 1024**2  # Convert to MB
print(memory_usage)
print(f"\nTotal Memory: {memory_usage.sum():.2f} MB")

### 2.2 Metadata Dataset Overview

In [ ]:
print("=" * 80)
print("METADATA DATASET OVERVIEW")
print("=" * 80)

print("\nDataset Shape:")
print(f"  Rows: {meta_df.shape[0]:,}")
print(f"  Columns: {meta_df.shape[1]}")

print("\nColumn Information:")
meta_df.info()

In [ ]:
print("\nFirst few rows:")
meta_df.head()

In [ ]:
print("\nBasic Statistics:")
meta_df.describe()

In [ ]:
# Memory usage
print("\nMemory Usage:")
memory_usage = meta_df.memory_usage(deep=True) / 1024**2  # Convert to MB
print(memory_usage)
print(f"\nTotal Memory: {memory_usage.sum():.2f} MB")

## 3. Missing Values Analysis

### 3.1 Review Dataset Missing Values

In [ ]:
print("=" * 80)
print("REVIEW DATASET - MISSING VALUES ANALYSIS")
print("=" * 80)

missing_review = pd.DataFrame({
    'Column': review_df.columns,
    'Missing_Count': review_df.isnull().sum(),
    'Missing_Percentage': (review_df.isnull().sum() / len(review_df) * 100)
}).sort_values('Missing_Percentage', ascending=False)

print(missing_review)

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
missing_review.set_index('Column')['Missing_Percentage'].plot(kind='barh', ax=ax)
ax.set_xlabel('Missing Percentage (%)')
ax.set_title('Missing Values in Review Dataset')
plt.tight_layout()
plt.show()

### 3.2 Metadata Dataset Missing Values

In [ ]:
print("=" * 80)
print("METADATA DATASET - MISSING VALUES ANALYSIS")
print("=" * 80)

missing_meta = pd.DataFrame({
    'Column': meta_df.columns,
    'Missing_Count': meta_df.isnull().sum(),
    'Missing_Percentage': (meta_df.isnull().sum() / len(meta_df) * 100)
}).sort_values('Missing_Percentage', ascending=False)

print(missing_meta)

# Visualize
fig, ax = plt.subplots(figsize=(10, 8))
missing_meta.set_index('Column')['Missing_Percentage'].plot(kind='barh', ax=ax)
ax.set_xlabel('Missing Percentage (%)')
ax.set_title('Missing Values in Metadata Dataset')
plt.tight_layout()
plt.show()

## 4. Deep Review Dataset Analysis

### 4.1 Rating Analysis

In [ ]:
print("=" * 80)
print("RATING DISTRIBUTION ANALYSIS")
print("=" * 80)

print("\nRating Statistics:")
print(review_df['rating'].describe())

print("\nRating Value Counts:")
rating_counts = review_df['rating'].value_counts().sort_index()
rating_pct = (rating_counts / len(review_df) * 100).round(2)
rating_summary = pd.DataFrame({
    'Count': rating_counts,
    'Percentage': rating_pct
})
print(rating_summary)

# Visualizations
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bar plot
rating_counts.plot(kind='bar', ax=axes[0], color='skyblue', edgecolor='black')
axes[0].set_title('Rating Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Add percentage labels on bars
for i, (count, pct) in enumerate(zip(rating_counts.values, rating_pct.values)):
    axes[0].text(i, count, f'{pct:.1f}%', ha='center', va='bottom')

# Pie chart
axes[1].pie(rating_counts.values, labels=rating_counts.index, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Rating Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

# Check for unusual ratings
print("\nUnique rating values:")
print(sorted(review_df['rating'].unique()))

In [ ]:
# Rating polarization
print("\nRating Polarization:")
extreme_ratings = review_df[review_df['rating'].isin([1.0, 5.0])]
print(f"Extreme ratings (1 or 5): {len(extreme_ratings):,} ({len(extreme_ratings)/len(review_df)*100:.2f}%)")

neutral_ratings = review_df[review_df['rating'] == 3.0]
print(f"Neutral ratings (3): {len(neutral_ratings):,} ({len(neutral_ratings)/len(review_df)*100:.2f}%)")

### 4.2 Temporal Analysis

In [ ]:
print("=" * 80)
print("TEMPORAL ANALYSIS")
print("=" * 80)

# Convert timestamp to datetime
review_df['datetime'] = pd.to_datetime(review_df['timestamp'], unit='ms')
review_df['year'] = review_df['datetime'].dt.year
review_df['month'] = review_df['datetime'].dt.month
review_df['day_of_week'] = review_df['datetime'].dt.day_name()
review_df['hour'] = review_df['datetime'].dt.hour

print("\nDate Range:")
print(f"  Earliest review: {review_df['datetime'].min()}")
print(f"  Latest review: {review_df['datetime'].max()}")
print(f"  Time span: {(review_df['datetime'].max() - review_df['datetime'].min()).days} days")

In [ ]:
# Reviews per year
print("\nReviews per Year:")
yearly_reviews = review_df['year'].value_counts().sort_index()
print(yearly_reviews)

fig, ax = plt.subplots(figsize=(12, 6))
yearly_reviews.plot(kind='bar', ax=ax, color='coral', edgecolor='black')
ax.set_title('Number of Reviews per Year', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Number of Reviews')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Reviews by month (overall pattern)
print("\nReviews by Month (Aggregated):")
monthly_reviews = review_df['month'].value_counts().sort_index()
print(monthly_reviews)

fig, ax = plt.subplots(figsize=(12, 6))
monthly_reviews.plot(kind='bar', ax=ax, color='lightgreen', edgecolor='black')
ax.set_title('Number of Reviews by Month', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Number of Reviews')
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
plt.tight_layout()
plt.show()

In [ ]:
# Reviews by day of week
print("\nReviews by Day of Week:")
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_reviews = review_df['day_of_week'].value_counts().reindex(day_order)
print(dow_reviews)

fig, ax = plt.subplots(figsize=(12, 6))
dow_reviews.plot(kind='bar', ax=ax, color='lightblue', edgecolor='black')
ax.set_title('Number of Reviews by Day of Week', fontsize=14, fontweight='bold')
ax.set_xlabel('Day of Week')
ax.set_ylabel('Number of Reviews')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Reviews by hour
print("\nReviews by Hour of Day:")
hourly_reviews = review_df['hour'].value_counts().sort_index()
print(hourly_reviews)

fig, ax = plt.subplots(figsize=(14, 6))
hourly_reviews.plot(kind='line', ax=ax, marker='o', color='purple', linewidth=2)
ax.set_title('Number of Reviews by Hour of Day', fontsize=14, fontweight='bold')
ax.set_xlabel('Hour')
ax.set_ylabel('Number of Reviews')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Rating trends over time
print("\nAverage Rating by Year:")
avg_rating_by_year = review_df.groupby('year')['rating'].mean()
print(avg_rating_by_year)

fig, ax = plt.subplots(figsize=(12, 6))
avg_rating_by_year.plot(kind='line', ax=ax, marker='o', color='darkgreen', linewidth=2)
ax.set_title('Average Rating Trend Over Years', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Average Rating')
ax.set_ylim(0, 5.5)
ax.axhline(y=review_df['rating'].mean(), color='red', linestyle='--', label='Overall Average')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 4.3 Text Analysis

In [ ]:
print("=" * 80)
print("TEXT ANALYSIS - REVIEW TITLES")
print("=" * 80)

# Title length analysis
review_df['title_length'] = review_df['title'].fillna('').astype(str).str.len()
review_df['title_word_count'] = review_df['title'].fillna('').astype(str).str.split().str.len()

print("\nTitle Length Statistics:")
print(review_df['title_length'].describe())

print("\nTitle Word Count Statistics:")
print(review_df['title_word_count'].describe())

# Empty titles
empty_titles = review_df[review_df['title'].isna() | (review_df['title'] == '')]
print(f"\nEmpty titles: {len(empty_titles):,} ({len(empty_titles)/len(review_df)*100:.2f}%)")

In [ ]:
# Visualize title length distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Title character length
review_df['title_length'].hist(bins=50, ax=axes[0], color='skyblue', edgecolor='black')
axes[0].set_title('Distribution of Title Length (Characters)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Title Length')
axes[0].set_ylabel('Frequency')
axes[0].axvline(review_df['title_length'].median(), color='red', linestyle='--', label=f'Median: {review_df["title_length"].median():.0f}')
axes[0].legend()

# Title word count
review_df['title_word_count'].hist(bins=30, ax=axes[1], color='lightcoral', edgecolor='black')
axes[1].set_title('Distribution of Title Word Count', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Frequency')
axes[1].axvline(review_df['title_word_count'].median(), color='red', linestyle='--', label=f'Median: {review_df["title_word_count"].median():.0f}')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
print("=" * 80)
print("TEXT ANALYSIS - REVIEW TEXT")
print("=" * 80)

# Review text length analysis
review_df['text_length'] = review_df['text'].fillna('').astype(str).str.len()
review_df['text_word_count'] = review_df['text'].fillna('').astype(str).str.split().str.len()

print("\nReview Text Length Statistics:")
print(review_df['text_length'].describe())

print("\nReview Text Word Count Statistics:")
print(review_df['text_word_count'].describe())

# Empty review text
empty_text = review_df[review_df['text'].isna() | (review_df['text'] == '')]
print(f"\nEmpty review text: {len(empty_text):,} ({len(empty_text)/len(review_df)*100:.2f}%)")

In [ ]:
# Visualize review text length distribution
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Text character length - full range
review_df['text_length'].hist(bins=100, ax=axes[0, 0], color='lightgreen', edgecolor='black')
axes[0, 0].set_title('Distribution of Review Text Length (Characters)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Text Length')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(review_df['text_length'].median(), color='red', linestyle='--', label=f'Median: {review_df["text_length"].median():.0f}')
axes[0, 0].legend()

# Text character length - zoomed (0-1000)
review_df[review_df['text_length'] <= 1000]['text_length'].hist(bins=50, ax=axes[0, 1], color='lightgreen', edgecolor='black')
axes[0, 1].set_title('Distribution of Review Text Length (0-1000 chars)', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Text Length')
axes[0, 1].set_ylabel('Frequency')

# Text word count - full range
review_df['text_word_count'].hist(bins=100, ax=axes[1, 0], color='lightyellow', edgecolor='black')
axes[1, 0].set_title('Distribution of Review Text Word Count', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Word Count')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].axvline(review_df['text_word_count'].median(), color='red', linestyle='--', label=f'Median: {review_df["text_word_count"].median():.0f}')
axes[1, 0].legend()

# Text word count - zoomed (0-200)
review_df[review_df['text_word_count'] <= 200]['text_word_count'].hist(bins=50, ax=axes[1, 1], color='lightyellow', edgecolor='black')
axes[1, 1].set_title('Distribution of Review Text Word Count (0-200 words)', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Word Count')
axes[1, 1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Text length by rating
print("\nAverage Text Length by Rating:")
text_by_rating = review_df.groupby('rating').agg({
    'text_length': 'mean',
    'text_word_count': 'mean'
}).round(2)
print(text_by_rating)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

text_by_rating['text_length'].plot(kind='bar', ax=axes[0], color='teal', edgecolor='black')
axes[0].set_title('Average Text Length by Rating', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Average Character Count')
axes[0].tick_params(axis='x', rotation=0)

text_by_rating['text_word_count'].plot(kind='bar', ax=axes[1], color='olive', edgecolor='black')
axes[1].set_title('Average Word Count by Rating', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Rating')
axes[1].set_ylabel('Average Word Count')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

### 4.4 User Behavior Analysis

In [ ]:
print("=" * 80)
print("USER BEHAVIOR ANALYSIS")
print("=" * 80)

# Unique users
unique_users = review_df['user_id'].nunique()
print(f"\nTotal unique users: {unique_users:,}")
print(f"Total reviews: {len(review_df):,}")
print(f"Average reviews per user: {len(review_df)/unique_users:.2f}")

In [ ]:
# Reviews per user distribution
reviews_per_user = review_df['user_id'].value_counts()

print("\nReviews per User Statistics:")
print(reviews_per_user.describe())

print("\nTop 10 Most Active Users:")
print(reviews_per_user.head(10))

# Distribution of review counts
print("\nDistribution of Users by Number of Reviews:")
user_review_dist = reviews_per_user.value_counts().sort_index().head(20)
print(user_review_dist)

In [ ]:
# Visualize user activity
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram of reviews per user (log scale)
axes[0].hist(reviews_per_user.values, bins=100, color='purple', edgecolor='black', log=True)
axes[0].set_title('Distribution of Reviews per User (Log Scale)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Number of Reviews')
axes[0].set_ylabel('Number of Users (log)')
axes[0].axvline(reviews_per_user.median(), color='red', linestyle='--', label=f'Median: {reviews_per_user.median():.0f}')
axes[0].legend()

# Top 20 most active users
reviews_per_user.head(20).plot(kind='bar', ax=axes[1], color='orange', edgecolor='black')
axes[1].set_title('Top 20 Most Active Users', fontsize=12, fontweight='bold')
axes[1].set_xlabel('User ID')
axes[1].set_ylabel('Number of Reviews')
axes[1].tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.show()

In [ ]:
# One-time reviewers vs repeat reviewers
one_time_reviewers = (reviews_per_user == 1).sum()
repeat_reviewers = (reviews_per_user > 1).sum()

print(f"\nOne-time reviewers: {one_time_reviewers:,} ({one_time_reviewers/unique_users*100:.2f}%)")
print(f"Repeat reviewers: {repeat_reviewers:,} ({repeat_reviewers/unique_users*100:.2f}%)")

# Power users (>10 reviews)
power_users = (reviews_per_user > 10).sum()
power_user_reviews = reviews_per_user[reviews_per_user > 10].sum()
print(f"\nPower users (>10 reviews): {power_users:,} ({power_users/unique_users*100:.2f}%)")
print(f"Reviews by power users: {power_user_reviews:,} ({power_user_reviews/len(review_df)*100:.2f}%)")

In [ ]:
# Verified purchase analysis
print("\nVerified Purchase Analysis:")
verified_counts = review_df['verified_purchase'].value_counts()
print(verified_counts)
print(f"\nVerified purchase rate: {verified_counts.get(True, 0)/len(review_df)*100:.2f}%")

# Rating by verified purchase
print("\nAverage Rating by Verified Purchase:")
rating_by_verified = review_df.groupby('verified_purchase')['rating'].agg(['mean', 'median', 'count'])
print(rating_by_verified)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Pie chart
verified_counts.plot(kind='pie', ax=axes[0], autopct='%1.1f%%', labels=['Not Verified', 'Verified'])
axes[0].set_title('Verified Purchase Distribution', fontsize=12, fontweight='bold')
axes[0].set_ylabel('')

# Rating comparison
rating_by_verified['mean'].plot(kind='bar', ax=axes[1], color=['coral', 'lightgreen'], edgecolor='black')
axes[1].set_title('Average Rating: Verified vs Not Verified', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Verified Purchase')
axes[1].set_ylabel('Average Rating')
axes[1].set_xticklabels(['Not Verified', 'Verified'], rotation=0)
axes[1].set_ylim(0, 5)

plt.tight_layout()
plt.show()

### 4.5 Helpful Votes Analysis

In [ ]:
print("=" * 80)
print("HELPFUL VOTES ANALYSIS")
print("=" * 80)

print("\nHelpful Votes Statistics:")
print(review_df['helpful_vote'].describe())

# Distribution
reviews_with_votes = review_df[review_df['helpful_vote'] > 0]
print(f"\nReviews with helpful votes: {len(reviews_with_votes):,} ({len(reviews_with_votes)/len(review_df)*100:.2f}%)")
print(f"Reviews with no votes: {len(review_df[review_df['helpful_vote'] == 0]):,} ({len(review_df[review_df['helpful_vote'] == 0])/len(review_df)*100:.2f}%)")

print("\nTop 10 Most Helpful Reviews:")
top_helpful = review_df.nlargest(10, 'helpful_vote')[['title', 'rating', 'helpful_vote', 'text_length']]
print(top_helpful)

In [ ]:
# Visualize helpful votes
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Distribution (all)
review_df['helpful_vote'].hist(bins=100, ax=axes[0, 0], color='skyblue', edgecolor='black', log=True)
axes[0, 0].set_title('Distribution of Helpful Votes (Log Scale)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Helpful Votes')
axes[0, 0].set_ylabel('Frequency (log)')

# Distribution (votes > 0, capped at 50)
reviews_with_votes[reviews_with_votes['helpful_vote'] <= 50]['helpful_vote'].hist(bins=50, ax=axes[0, 1], color='lightcoral', edgecolor='black')
axes[0, 1].set_title('Distribution of Helpful Votes (1-50 votes)', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Helpful Votes')
axes[0, 1].set_ylabel('Frequency')

# Helpful votes by rating
helpful_by_rating = review_df.groupby('rating')['helpful_vote'].mean()
helpful_by_rating.plot(kind='bar', ax=axes[1, 0], color='lightgreen', edgecolor='black')
axes[1, 0].set_title('Average Helpful Votes by Rating', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Rating')
axes[1, 0].set_ylabel('Average Helpful Votes')
axes[1, 0].tick_params(axis='x', rotation=0)

# Helpful votes vs text length
axes[1, 1].scatter(review_df['text_length'], review_df['helpful_vote'], alpha=0.1, s=1)
axes[1, 1].set_title('Helpful Votes vs Review Text Length', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Review Text Length')
axes[1, 1].set_ylabel('Helpful Votes')
axes[1, 1].set_xlim(0, 2000)
axes[1, 1].set_ylim(0, 100)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation between helpful votes and other features
print("\nCorrelation with Helpful Votes:")
correlation_features = ['rating', 'text_length', 'text_word_count', 'helpful_vote']
correlation = review_df[correlation_features].corr()['helpful_vote'].sort_values(ascending=False)
print(correlation)

### 4.6 Product (ASIN) Analysis

In [ ]:
print("=" * 80)
print("PRODUCT (ASIN) ANALYSIS")
print("=" * 80)

# Unique products
unique_asins = review_df['asin'].nunique()
unique_parent_asins = review_df['parent_asin'].nunique()

print(f"\nUnique ASINs: {unique_asins:,}")
print(f"Unique Parent ASINs: {unique_parent_asins:,}")
print(f"Average reviews per ASIN: {len(review_df)/unique_asins:.2f}")
print(f"Average reviews per Parent ASIN: {len(review_df)/unique_parent_asins:.2f}")

In [ ]:
# Reviews per product
reviews_per_asin = review_df['parent_asin'].value_counts()

print("\nReviews per Product (Parent ASIN) Statistics:")
print(reviews_per_asin.describe())

print("\nTop 10 Most Reviewed Products:")
print(reviews_per_asin.head(10))

# Products with only one review
single_review_products = (reviews_per_asin == 1).sum()
print(f"\nProducts with only 1 review: {single_review_products:,} ({single_review_products/unique_parent_asins*100:.2f}%)")

# Products with >100 reviews
popular_products = (reviews_per_asin > 100).sum()
popular_products_reviews = reviews_per_asin[reviews_per_asin > 100].sum()
print(f"\nPopular products (>100 reviews): {popular_products:,} ({popular_products/unique_parent_asins*100:.2f}%)")
print(f"Reviews for popular products: {popular_products_reviews:,} ({popular_products_reviews/len(review_df)*100:.2f}%)")

In [ ]:
# Visualize product review distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram (log scale)
axes[0].hist(reviews_per_asin.values, bins=100, color='teal', edgecolor='black', log=True)
axes[0].set_title('Distribution of Reviews per Product (Log Scale)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Number of Reviews')
axes[0].set_ylabel('Number of Products (log)')
axes[0].axvline(reviews_per_asin.median(), color='red', linestyle='--', label=f'Median: {reviews_per_asin.median():.0f}')
axes[0].legend()

# Top 20 products
reviews_per_asin.head(20).plot(kind='bar', ax=axes[1], color='coral', edgecolor='black')
axes[1].set_title('Top 20 Most Reviewed Products', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Parent ASIN')
axes[1].set_ylabel('Number of Reviews')
axes[1].tick_params(axis='x', rotation=90)

plt.tight_layout()
plt.show()

In [ ]:
# Average rating per product
product_ratings = review_df.groupby('parent_asin').agg({
    'rating': ['mean', 'std', 'count']
}).round(2)
product_ratings.columns = ['avg_rating', 'rating_std', 'review_count']
product_ratings = product_ratings.sort_values('review_count', ascending=False)

print("\nTop 10 Products by Review Count (with ratings):")
print(product_ratings.head(10))

print("\nProducts with highest average rating (min 50 reviews):")
high_rated = product_ratings[product_ratings['review_count'] >= 50].sort_values('avg_rating', ascending=False)
print(high_rated.head(10))

### 4.7 Images in Reviews

In [ ]:
print("=" * 80)
print("REVIEW IMAGES ANALYSIS")
print("=" * 80)

# Count reviews with images
review_df['has_images'] = review_df['images'].apply(lambda x: len(x) > 0 if isinstance(x, list) else False)
review_df['image_count'] = review_df['images'].apply(lambda x: len(x) if isinstance(x, list) else 0)

reviews_with_images = review_df[review_df['has_images']]
print(f"\nReviews with images: {len(reviews_with_images):,} ({len(reviews_with_images)/len(review_df)*100:.2f}%)")
print(f"Reviews without images: {len(review_df[~review_df['has_images']]):,} ({len(review_df[~review_df['has_images']])/len(review_df)*100:.2f}%)")

print("\nImage Count Statistics (for reviews with images):")
print(reviews_with_images['image_count'].describe())

if len(reviews_with_images) > 0:
    print("\nImage Count Distribution:")
    print(reviews_with_images['image_count'].value_counts().sort_index())

## 5. Deep Metadata Dataset Analysis

### 5.1 Category Analysis

In [ ]:
print("=" * 80)
print("CATEGORY ANALYSIS")
print("=" * 80)

# Main category distribution
print("\nMain Category Distribution:")
main_category_counts = meta_df['main_category'].value_counts()
print(main_category_counts)

fig, ax = plt.subplots(figsize=(12, 6))
main_category_counts.plot(kind='bar', ax=ax, color='skyblue', edgecolor='black')
ax.set_title('Distribution of Main Categories', fontsize=14, fontweight='bold')
ax.set_xlabel('Main Category')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Detailed categories analysis
print("\nDetailed Categories Analysis:")

# Count products with categories
meta_df['has_categories'] = meta_df['categories'].apply(lambda x: len(x) > 0 if isinstance(x, list) else False)
meta_df['category_depth'] = meta_df['categories'].apply(lambda x: len(x) if isinstance(x, list) else 0)

products_with_cats = meta_df[meta_df['has_categories']]
print(f"\nProducts with categories: {len(products_with_cats):,} ({len(products_with_cats)/len(meta_df)*100:.2f}%)")

if len(products_with_cats) > 0:
    print("\nCategory Depth Statistics:")
    print(products_with_cats['category_depth'].describe())
    
    print("\nCategory Depth Distribution:")
    print(products_with_cats['category_depth'].value_counts().sort_index())

### 5.2 Price Analysis

In [ ]:
print("=" * 80)
print("PRICE ANALYSIS")
print("=" * 80)

# Price statistics
price_data = meta_df[meta_df['price'].notna() & (meta_df['price'] > 0)]
print(f"\nProducts with price information: {len(price_data):,} ({len(price_data)/len(meta_df)*100:.2f}%)")
print(f"Products without price: {len(meta_df[meta_df['price'].isna() | (meta_df['price'] == 0)]):,}")

if len(price_data) > 0:
    print("\nPrice Statistics:")
    print(price_data['price'].describe())
    
    # Price ranges
    print("\nPrice Ranges:")
    price_ranges = pd.cut(price_data['price'], bins=[0, 10, 25, 50, 100, 250, 500, 1000, float('inf')],
                          labels=['$0-10', '$10-25', '$25-50', '$50-100', '$100-250', '$250-500', '$500-1000', '$1000+'])
    print(price_ranges.value_counts().sort_index())

In [ ]:
# Visualize price distribution
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Full price distribution
price_data['price'].hist(bins=100, ax=axes[0, 0], color='lightgreen', edgecolor='black', log=True)
axes[0, 0].set_title('Price Distribution (Log Scale)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Price ($)')
axes[0, 0].set_ylabel('Frequency (log)')

# Price distribution (0-100)
price_data[price_data['price'] <= 100]['price'].hist(bins=50, ax=axes[0, 1], color='lightgreen', edgecolor='black')
axes[0, 1].set_title('Price Distribution ($0-$100)', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Price ($)')
axes[0, 1].set_ylabel('Frequency')

# Box plot
axes[1, 0].boxplot(price_data['price'], vert=True)
axes[1, 0].set_title('Price Box Plot', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Price ($)')

# Price ranges bar chart
price_ranges.value_counts().sort_index().plot(kind='bar', ax=axes[1, 1], color='coral', edgecolor='black')
axes[1, 1].set_title('Distribution by Price Range', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Price Range')
axes[1, 1].set_ylabel('Number of Products')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Free products
free_products = meta_df[meta_df['price'] == 0]
print(f"\nFree products (price = 0): {len(free_products):,} ({len(free_products)/len(meta_df)*100:.2f}%)")

# Most expensive products
print("\nTop 10 Most Expensive Products:")
expensive = meta_df.nlargest(10, 'price')[['title', 'price', 'average_rating', 'rating_number']]
print(expensive)

### 5.3 Rating Analysis (Metadata)

In [ ]:
print("=" * 80)
print("RATING ANALYSIS (METADATA)")
print("=" * 80)

# Average rating statistics
rating_data = meta_df[meta_df['average_rating'].notna()]
print(f"\nProducts with average rating: {len(rating_data):,} ({len(rating_data)/len(meta_df)*100:.2f}%)")

if len(rating_data) > 0:
    print("\nAverage Rating Statistics:")
    print(rating_data['average_rating'].describe())
    
    # Rating number statistics
    rating_num_data = meta_df[meta_df['rating_number'].notna()]
    print(f"\nProducts with rating number: {len(rating_num_data):,} ({len(rating_num_data)/len(meta_df)*100:.2f}%)")
    
    if len(rating_num_data) > 0:
        print("\nRating Number (Count) Statistics:")
        print(rating_num_data['rating_number'].describe())

In [ ]:
# Visualize metadata ratings
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Average rating distribution
rating_data['average_rating'].hist(bins=50, ax=axes[0, 0], color='skyblue', edgecolor='black')
axes[0, 0].set_title('Distribution of Average Ratings', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Average Rating')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(rating_data['average_rating'].median(), color='red', linestyle='--', label=f'Median: {rating_data["average_rating"].median():.2f}')
axes[0, 0].legend()

# Rating number distribution (log scale)
if len(rating_num_data) > 0:
    rating_num_data['rating_number'].hist(bins=100, ax=axes[0, 1], color='lightcoral', edgecolor='black', log=True)
    axes[0, 1].set_title('Distribution of Rating Counts (Log Scale)', fontsize=12, fontweight='bold')
    axes[0, 1].set_xlabel('Number of Ratings')
    axes[0, 1].set_ylabel('Frequency (log)')

# Average rating by main category
category_ratings = meta_df.groupby('main_category')['average_rating'].mean().sort_values()
category_ratings.plot(kind='barh', ax=axes[1, 0], color='lightgreen', edgecolor='black')
axes[1, 0].set_title('Average Rating by Main Category', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Average Rating')
axes[1, 0].set_ylabel('Category')

# Scatter: rating vs price
if len(price_data) > 0 and len(rating_data) > 0:
    combined_data = meta_df[(meta_df['price'].notna()) & (meta_df['price'] > 0) & (meta_df['average_rating'].notna())]
    axes[1, 1].scatter(combined_data['price'], combined_data['average_rating'], alpha=0.3, s=10)
    axes[1, 1].set_title('Average Rating vs Price', fontsize=12, fontweight='bold')
    axes[1, 1].set_xlabel('Price ($)')
    axes[1, 1].set_ylabel('Average Rating')
    axes[1, 1].set_xlim(0, 500)

plt.tight_layout()
plt.show()

### 5.4 Store Analysis

In [ ]:
print("=" * 80)
print("STORE ANALYSIS")
print("=" * 80)

# Store statistics
store_data = meta_df[meta_df['store'].notna()]
print(f"\nProducts with store information: {len(store_data):,} ({len(store_data)/len(meta_df)*100:.2f}%)")

if len(store_data) > 0:
    unique_stores = store_data['store'].nunique()
    print(f"Unique stores: {unique_stores:,}")
    
    # Products per store
    products_per_store = store_data['store'].value_counts()
    print("\nProducts per Store Statistics:")
    print(products_per_store.describe())
    
    print("\nTop 20 Stores by Product Count:")
    print(products_per_store.head(20))

In [ ]:
# Visualize top stores
if len(store_data) > 0:
    fig, ax = plt.subplots(figsize=(12, 8))
    products_per_store.head(20).plot(kind='barh', ax=ax, color='purple', edgecolor='black')
    ax.set_title('Top 20 Stores by Product Count', fontsize=14, fontweight='bold')
    ax.set_xlabel('Number of Products')
    ax.set_ylabel('Store')
    plt.tight_layout()
    plt.show()
    
    # Store concentration
    top_10_stores_products = products_per_store.head(10).sum()
    print(f"\nProducts from top 10 stores: {top_10_stores_products:,} ({top_10_stores_products/len(store_data)*100:.2f}%)")

### 5.5 Features and Description Analysis

In [ ]:
print("=" * 80)
print("FEATURES ANALYSIS")
print("=" * 80)

# Features analysis
meta_df['has_features'] = meta_df['features'].apply(lambda x: len(x) > 0 if isinstance(x, list) else False)
meta_df['feature_count'] = meta_df['features'].apply(lambda x: len(x) if isinstance(x, list) else 0)

products_with_features = meta_df[meta_df['has_features']]
print(f"\nProducts with features: {len(products_with_features):,} ({len(products_with_features)/len(meta_df)*100:.2f}%)")

if len(products_with_features) > 0:
    print("\nFeature Count Statistics:")
    print(products_with_features['feature_count'].describe())
    
    print("\nFeature Count Distribution:")
    print(products_with_features['feature_count'].value_counts().sort_index().head(20))

In [ ]:
print("\n" + "=" * 80)
print("DESCRIPTION ANALYSIS")
print("=" * 80)

# Description analysis
meta_df['has_description'] = meta_df['description'].apply(lambda x: len(x) > 0 if isinstance(x, list) else False)
meta_df['description_count'] = meta_df['description'].apply(lambda x: len(x) if isinstance(x, list) else 0)

products_with_desc = meta_df[meta_df['has_description']]
print(f"\nProducts with description: {len(products_with_desc):,} ({len(products_with_desc)/len(meta_df)*100:.2f}%)")

if len(products_with_desc) > 0:
    print("\nDescription Count Statistics:")
    print(products_with_desc['description_count'].describe())
    
    # Calculate total description length
    meta_df['description_length'] = meta_df['description'].apply(
        lambda x: sum(len(str(item)) for item in x) if isinstance(x, list) else 0
    )
    
    print("\nDescription Length Statistics:")
    print(products_with_desc[products_with_desc['description_length'] > 0]['description_length'].describe())

In [ ]:
# Visualize features and descriptions
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Feature count distribution
if len(products_with_features) > 0:
    products_with_features['feature_count'].hist(bins=30, ax=axes[0, 0], color='skyblue', edgecolor='black')
    axes[0, 0].set_title('Distribution of Feature Count', fontsize=12, fontweight='bold')
    axes[0, 0].set_xlabel('Number of Features')
    axes[0, 0].set_ylabel('Frequency')

# Description count distribution
if len(products_with_desc) > 0:
    products_with_desc['description_count'].hist(bins=30, ax=axes[0, 1], color='lightcoral', edgecolor='black')
    axes[0, 1].set_title('Distribution of Description Count', fontsize=12, fontweight='bold')
    axes[0, 1].set_xlabel('Number of Description Items')
    axes[0, 1].set_ylabel('Frequency')

# Description length distribution
desc_length_data = meta_df[meta_df['description_length'] > 0]
if len(desc_length_data) > 0:
    desc_length_data['description_length'].hist(bins=50, ax=axes[1, 0], color='lightgreen', edgecolor='black')
    axes[1, 0].set_title('Distribution of Total Description Length', fontsize=12, fontweight='bold')
    axes[1, 0].set_xlabel('Total Description Length (characters)')
    axes[1, 0].set_ylabel('Frequency')

# Products with/without features and descriptions
content_summary = pd.DataFrame({
    'Has Features': [len(products_with_features), len(meta_df) - len(products_with_features)],
    'Has Description': [len(products_with_desc), len(meta_df) - len(products_with_desc)]
}, index=['Yes', 'No'])
content_summary.plot(kind='bar', ax=axes[1, 1], color=['skyblue', 'lightcoral'], edgecolor='black')
axes[1, 1].set_title('Products with Features/Descriptions', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Has Content')
axes[1, 1].set_ylabel('Number of Products')
axes[1, 1].tick_params(axis='x', rotation=0)
axes[1, 1].legend()

plt.tight_layout()
plt.show()

### 5.6 Product Details Analysis

In [ ]:
print("=" * 80)
print("PRODUCT DETAILS ANALYSIS")
print("=" * 80)

# Details analysis
meta_df['has_details'] = meta_df['details'].apply(lambda x: len(x) > 0 if isinstance(x, dict) else False)
meta_df['details_count'] = meta_df['details'].apply(lambda x: len(x) if isinstance(x, dict) else 0)

products_with_details = meta_df[meta_df['has_details']]
print(f"\nProducts with details: {len(products_with_details):,} ({len(products_with_details)/len(meta_df)*100:.2f}%)")

if len(products_with_details) > 0:
    print("\nNumber of Detail Fields Statistics:")
    print(products_with_details['details_count'].describe())
    
    # Extract all unique detail keys
    all_keys = set()
    for details in products_with_details['details']:
        if isinstance(details, dict):
            all_keys.update(details.keys())
    
    print(f"\nUnique detail field types: {len(all_keys)}")
    
    # Count frequency of each detail key
    key_counts = {}
    for details in products_with_details['details']:
        if isinstance(details, dict):
            for key in details.keys():
                key_counts[key] = key_counts.get(key, 0) + 1
    
    key_counts_df = pd.Series(key_counts).sort_values(ascending=False)
    print("\nTop 20 Most Common Detail Fields:")
    print(key_counts_df.head(20))

In [ ]:
# Visualize details
if len(products_with_details) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Details count distribution
    products_with_details['details_count'].hist(bins=30, ax=axes[0], color='purple', edgecolor='black')
    axes[0].set_title('Distribution of Detail Field Count', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Number of Detail Fields')
    axes[0].set_ylabel('Frequency')
    
    # Top detail fields
    key_counts_df.head(15).plot(kind='barh', ax=axes[1], color='orange', edgecolor='black')
    axes[1].set_title('Top 15 Most Common Detail Fields', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Frequency')
    axes[1].set_ylabel('Detail Field')
    
    plt.tight_layout()
    plt.show()

### 5.7 Images and Videos in Metadata

In [ ]:
print("=" * 80)
print("PRODUCT MEDIA ANALYSIS")
print("=" * 80)

# Images analysis
def count_images(img_dict):
    if not isinstance(img_dict, dict):
        return 0
    # Count non-None images in any size category
    count = 0
    for size_key in ['hi_res', 'large', 'thumb']:
        if size_key in img_dict and isinstance(img_dict[size_key], list):
            count = max(count, sum(1 for img in img_dict[size_key] if img is not None))
    return count

meta_df['image_count'] = meta_df['images'].apply(count_images)
meta_df['has_images'] = meta_df['image_count'] > 0

products_with_images = meta_df[meta_df['has_images']]
print(f"\nProducts with images: {len(products_with_images):,} ({len(products_with_images)/len(meta_df)*100:.2f}%)")

if len(products_with_images) > 0:
    print("\nImage Count Statistics:")
    print(products_with_images['image_count'].describe())
    
    print("\nImage Count Distribution:")
    print(products_with_images['image_count'].value_counts().sort_index().head(15))

In [ ]:
# Videos analysis
def count_videos(vid_dict):
    if not isinstance(vid_dict, dict):
        return 0
    if 'url' in vid_dict and isinstance(vid_dict['url'], list):
        return sum(1 for url in vid_dict['url'] if url and url != '')
    return 0

meta_df['video_count'] = meta_df['videos'].apply(count_videos)
meta_df['has_videos'] = meta_df['video_count'] > 0

products_with_videos = meta_df[meta_df['has_videos']]
print("\nProducts with videos: {:,} ({:.2f}%)".format(
    len(products_with_videos), len(products_with_videos)/len(meta_df)*100
))

if len(products_with_videos) > 0:
    print("\nVideo Count Statistics:")
    print(products_with_videos['video_count'].describe())
    
    print("\nVideo Count Distribution:")
    print(products_with_videos['video_count'].value_counts().sort_index())

In [ ]:
# Visualize media
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Image count distribution
if len(products_with_images) > 0:
    products_with_images['image_count'].hist(bins=20, ax=axes[0], color='teal', edgecolor='black')
    axes[0].set_title('Distribution of Image Count per Product', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Number of Images')
    axes[0].set_ylabel('Frequency')

# Media availability summary
media_summary = pd.DataFrame({
    'Images': [len(products_with_images), len(meta_df) - len(products_with_images)],
    'Videos': [len(products_with_videos), len(meta_df) - len(products_with_videos)]
}, index=['Has Media', 'No Media'])
media_summary.plot(kind='bar', ax=axes[1], color=['teal', 'coral'], edgecolor='black')
axes[1].set_title('Products with Images/Videos', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Media Type')
axes[1].set_ylabel('Number of Products')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend()

plt.tight_layout()
plt.show()

### 5.8 Other Metadata Fields

In [ ]:
print("=" * 80)
print("OTHER METADATA FIELDS")
print("=" * 80)

# Title analysis
meta_df['title_length'] = meta_df['title'].fillna('').astype(str).str.len()
meta_df['title_word_count'] = meta_df['title'].fillna('').astype(str).str.split().str.len()

print("\nProduct Title Length Statistics:")
print(meta_df['title_length'].describe())

print("\nProduct Title Word Count Statistics:")
print(meta_df['title_word_count'].describe())

# Bought together analysis
meta_df['has_bought_together'] = meta_df['bought_together'].apply(
    lambda x: len(x) > 0 if isinstance(x, list) else False
)
products_with_bundles = meta_df[meta_df['has_bought_together']]
print(f"\nProducts with 'bought together' recommendations: {len(products_with_bundles):,} ({len(products_with_bundles)/len(meta_df)*100:.2f}%)")

# Subtitle and author
products_with_subtitle = meta_df[meta_df['subtitle'].notna()]
products_with_author = meta_df[meta_df['author'].notna()]

print(f"\nProducts with subtitle: {len(products_with_subtitle):,} ({len(products_with_subtitle)/len(meta_df)*100:.2f}%)")
print(f"Products with author: {len(products_with_author):,} ({len(products_with_author)/len(meta_df)*100:.2f}%)")

## 6. Relationship Analysis Between Datasets

### 6.1 Dataset Join and Coverage

In [ ]:
print("=" * 80)
print("DATASET RELATIONSHIP ANALYSIS")
print("=" * 80)

# Check overlap between review parent_asin and meta parent_asin
review_parent_asins = set(review_df['parent_asin'].unique())
meta_parent_asins = set(meta_df['parent_asin'].unique())

print(f"\nUnique parent_asins in reviews: {len(review_parent_asins):,}")
print(f"Unique parent_asins in metadata: {len(meta_parent_asins):,}")

# Intersection and differences
common_asins = review_parent_asins.intersection(meta_parent_asins)
reviews_only = review_parent_asins - meta_parent_asins
meta_only = meta_parent_asins - review_parent_asins

print(f"\nCommon parent_asins: {len(common_asins):,} ({len(common_asins)/len(review_parent_asins)*100:.2f}% of review ASINs)")
print(f"Only in reviews (no metadata): {len(reviews_only):,} ({len(reviews_only)/len(review_parent_asins)*100:.2f}%)")
print(f"Only in metadata (no reviews): {len(meta_only):,} ({len(meta_only)/len(meta_parent_asins)*100:.2f}%)")

In [ ]:
# Reviews for products without metadata
reviews_without_meta = review_df[review_df['parent_asin'].isin(reviews_only)]
print(f"\nNumber of reviews for products without metadata: {len(reviews_without_meta):,} ({len(reviews_without_meta)/len(review_df)*100:.2f}%)")

# Products without reviews
meta_without_reviews = meta_df[meta_df['parent_asin'].isin(meta_only)]
print(f"Number of products without reviews: {len(meta_without_reviews):,} ({len(meta_without_reviews)/len(meta_df)*100:.2f}%)")

In [ ]:
# Join datasets
print("\nJoining datasets on parent_asin...")
merged_df = review_df.merge(meta_df, on='parent_asin', how='inner', suffixes=('_review', '_meta'))
print(f"Merged dataset size: {len(merged_df):,} rows")
print(f"Percentage of reviews successfully joined: {len(merged_df)/len(review_df)*100:.2f}%")

### 6.2 Rating Comparison

In [ ]:
print("=" * 80)
print("RATING COMPARISON (REVIEWS vs METADATA)")
print("=" * 80)

# Calculate average rating from reviews for each product
review_avg_ratings = review_df.groupby('parent_asin')['rating'].agg(['mean', 'count']).reset_index()
review_avg_ratings.columns = ['parent_asin', 'review_calculated_avg', 'review_count']

# Merge with metadata
rating_comparison = review_avg_ratings.merge(
    meta_df[['parent_asin', 'average_rating', 'rating_number']], 
    on='parent_asin', 
    how='inner'
)

# Filter valid comparisons
valid_comparison = rating_comparison[
    rating_comparison['average_rating'].notna() & 
    (rating_comparison['average_rating'] > 0)
]

print(f"\nProducts with both review ratings and metadata ratings: {len(valid_comparison):,}")

if len(valid_comparison) > 0:
    # Calculate difference
    valid_comparison['rating_diff'] = valid_comparison['review_calculated_avg'] - valid_comparison['average_rating']
    
    print("\nRating Difference Statistics (Review Avg - Meta Avg):")
    print(valid_comparison['rating_diff'].describe())
    
    # Correlation
    correlation = valid_comparison[['review_calculated_avg', 'average_rating']].corr()
    print("\nCorrelation between review average and metadata average:")
    print(f"  {correlation.iloc[0, 1]:.4f}")
    
    # Count comparison
    count_comparison = valid_comparison[
        valid_comparison['rating_number'].notna() & 
        (valid_comparison['rating_number'] > 0)
    ]
    
    if len(count_comparison) > 0:
        count_comparison['count_diff'] = count_comparison['review_count'] - count_comparison['rating_number']
        print("\nRating Count Difference Statistics (Review Count - Meta Count):")
        print(count_comparison['count_diff'].describe())

In [ ]:
# Visualize rating comparison
if len(valid_comparison) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # Scatter plot: review avg vs meta avg
    axes[0, 0].scatter(valid_comparison['average_rating'], 
                      valid_comparison['review_calculated_avg'], 
                      alpha=0.3, s=10)
    axes[0, 0].plot([1, 5], [1, 5], 'r--', label='Perfect Agreement')
    axes[0, 0].set_title('Review Avg Rating vs Metadata Avg Rating', fontsize=12, fontweight='bold')
    axes[0, 0].set_xlabel('Metadata Average Rating')
    axes[0, 0].set_ylabel('Review Calculated Average Rating')
    axes[0, 0].legend()
    axes[0, 0].set_xlim(0, 5.5)
    axes[0, 0].set_ylim(0, 5.5)
    
    # Rating difference distribution
    valid_comparison['rating_diff'].hist(bins=100, ax=axes[0, 1], color='coral', edgecolor='black')
    axes[0, 1].set_title('Distribution of Rating Differences', fontsize=12, fontweight='bold')
    axes[0, 1].set_xlabel('Rating Difference (Review - Meta)')
    axes[0, 1].set_ylabel('Frequency')
    axes[0, 1].axvline(0, color='red', linestyle='--', label='No Difference')
    axes[0, 1].legend()
    
    # Scatter plot: review count vs meta count
    if len(count_comparison) > 0:
        axes[1, 0].scatter(count_comparison['rating_number'], 
                          count_comparison['review_count'], 
                          alpha=0.3, s=10)
        max_count = max(count_comparison['rating_number'].max(), count_comparison['review_count'].max())
        axes[1, 0].plot([0, max_count], [0, max_count], 'r--', label='Perfect Agreement')
        axes[1, 0].set_title('Review Count vs Metadata Rating Number', fontsize=12, fontweight='bold')
        axes[1, 0].set_xlabel('Metadata Rating Number')
        axes[1, 0].set_ylabel('Review Count')
        axes[1, 0].legend()
        axes[1, 0].set_xlim(0, None)
        axes[1, 0].set_ylim(0, None)
    
    # Box plot of rating difference by rating range
    valid_comparison['rating_range'] = pd.cut(
        valid_comparison['average_rating'], 
        bins=[0, 2, 3, 4, 5], 
        labels=['1-2', '2-3', '3-4', '4-5']
    )
    valid_comparison.boxplot(column='rating_diff', by='rating_range', ax=axes[1, 1])
    axes[1, 1].set_title('Rating Difference by Meta Rating Range', fontsize=12, fontweight='bold')
    axes[1, 1].set_xlabel('Metadata Rating Range')
    axes[1, 1].set_ylabel('Rating Difference')
    plt.suptitle('')  # Remove the automatic title
    
    plt.tight_layout()
    plt.show()

### 6.3 Price vs Rating Analysis

In [ ]:
print("=" * 80)
print("PRICE vs RATING ANALYSIS")
print("=" * 80)

# Use merged dataset with valid prices and ratings
price_rating_data = merged_df[
    (merged_df['price'].notna()) & 
    (merged_df['price'] > 0) & 
    (merged_df['price'] < 1000)  # Filter extreme outliers
]

print(f"\nReviews with valid price data: {len(price_rating_data):,}")

if len(price_rating_data) > 0:
    # Correlation
    correlation = price_rating_data[['price', 'rating']].corr()
    print(f"\nCorrelation between price and review rating: {correlation.iloc[0, 1]:.4f}")
    
    # Average rating by price range
    price_rating_data['price_range'] = pd.cut(
        price_rating_data['price'], 
        bins=[0, 10, 25, 50, 100, 250, 500, 1000],
        labels=['$0-10', '$10-25', '$25-50', '$50-100', '$100-250', '$250-500', '$500-1000']
    )
    
    rating_by_price_range = price_rating_data.groupby('price_range')['rating'].agg(['mean', 'count'])
    print("\nAverage Rating by Price Range:")
    print(rating_by_price_range)

In [ ]:
# Visualize price vs rating
if len(price_rating_data) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Scatter plot
    axes[0].scatter(price_rating_data['price'], price_rating_data['rating'], alpha=0.1, s=1)
    axes[0].set_title('Price vs Review Rating', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Price ($)')
    axes[0].set_ylabel('Rating')
    axes[0].set_ylim(0, 5.5)
    
    # Average rating by price range
    rating_by_price_range['mean'].plot(kind='bar', ax=axes[1], color='teal', edgecolor='black')
    axes[1].set_title('Average Rating by Price Range', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Price Range')
    axes[1].set_ylabel('Average Rating')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].set_ylim(0, 5)
    axes[1].axhline(y=price_rating_data['rating'].mean(), color='red', linestyle='--', label='Overall Avg')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()

## 7. Data Quality Assessment

### 7.1 Duplicate Detection

In [ ]:
print("=" * 80)
print("DUPLICATE DETECTION")
print("=" * 80)

# Check for duplicate reviews
print("\nREVIEW DATASET:")
duplicate_reviews = review_df.duplicated().sum()
print(f"Fully duplicate rows: {duplicate_reviews:,}")

# Check for duplicate user-product combinations
duplicate_user_product = review_df.duplicated(subset=['user_id', 'parent_asin']).sum()
print(f"Duplicate user-product combinations: {duplicate_user_product:,} ({duplicate_user_product/len(review_df)*100:.2f}%)")

# Check for duplicate review text
duplicate_text = review_df.duplicated(subset=['text']).sum()
print(f"Duplicate review text: {duplicate_text:,} ({duplicate_text/len(review_df)*100:.2f}%)")

print("\n\nMETADATA DATASET:")
duplicate_meta = meta_df.duplicated().sum()
print(f"Fully duplicate rows: {duplicate_meta:,}")

# Check for duplicate parent_asin
duplicate_asin = meta_df.duplicated(subset=['parent_asin']).sum()
print(f"Duplicate parent_asin: {duplicate_asin:,}")

### 7.2 Data Completeness Summary

In [ ]:
print("=" * 80)
print("DATA COMPLETENESS SUMMARY")
print("=" * 80)

# Review dataset completeness
print("\nREVIEW DATASET COMPLETENESS:")
review_completeness = pd.DataFrame({
    'Column': review_df.columns,
    'Non-Null': review_df.count(),
    'Non-Null %': (review_df.count() / len(review_df) * 100).round(2)
}).sort_values('Non-Null %')
print(review_completeness)

print("\n\nMETADATA DATASET COMPLETENESS:")
meta_completeness = pd.DataFrame({
    'Column': meta_df.columns,
    'Non-Null': meta_df.count(),
    'Non-Null %': (meta_df.count() / len(meta_df) * 100).round(2)
}).sort_values('Non-Null %')
print(meta_completeness)

### 7.3 Data Quality Issues Summary

In [ ]:
print("=" * 80)
print("DATA QUALITY ISSUES SUMMARY")
print("=" * 80)

print("\nREVIEW DATASET ISSUES:")
print(f"  - Empty review titles: {len(review_df[review_df['title'].isna() | (review_df['title'] == '')]):,}")
print(f"  - Empty review text: {len(review_df[review_df['text'].isna() | (review_df['text'] == '')]):,}")
print(f"  - Duplicate user-product combinations: {duplicate_user_product:,}")
print(f"  - Reviews without matching metadata: {len(reviews_without_meta):,}")

print("\nMETADATA DATASET ISSUES:")
print(f"  - Missing prices: {len(meta_df[meta_df['price'].isna()]):,}")
print(f"  - Missing average ratings: {len(meta_df[meta_df['average_rating'].isna()]):,}")
print(f"  - Missing categories: {len(meta_df[~meta_df['has_categories']]):,}")
print(f"  - Missing features: {len(meta_df[~meta_df['has_features']]):,}")
print(f"  - Missing descriptions: {len(meta_df[~meta_df['has_description']]):,}")
print(f"  - Products without reviews: {len(meta_without_reviews):,}")

print("\nDATASET RELATIONSHIP ISSUES:")
print(f"  - Reviews without metadata: {len(reviews_only):,} parent_asins")
print(f"  - Products without reviews: {len(meta_only):,} parent_asins")
print(f"  - Coverage: {len(common_asins)/len(review_parent_asins)*100:.2f}% of reviewed products have metadata")

## 8. Key Insights and Recommendations

In [ ]:
print("=" * 80)
print("KEY INSIGHTS")
print("=" * 80)

print("""
REVIEW DATASET KEY INSIGHTS:
-------------------------------
""")

print(f"1. SCALE: {len(review_df):,} reviews from {review_df['user_id'].nunique():,} users on {review_df['parent_asin'].nunique():,} products")
print(f"\n2. RATING DISTRIBUTION: {(review_df['rating'] >= 4).sum()/len(review_df)*100:.1f}% are 4-5 stars (positive skew)")
print(f"\n3. VERIFIED PURCHASES: {review_df['verified_purchase'].sum()/len(review_df)*100:.1f}% are verified")
print(f"\n4. HELPFUL VOTES: Only {(review_df['helpful_vote'] > 0).sum()/len(review_df)*100:.1f}% of reviews have helpful votes")
print(f"\n5. USER ACTIVITY: {(reviews_per_user == 1).sum()/unique_users*100:.1f}% are one-time reviewers")
print(f"\n6. TEMPORAL RANGE: Reviews span from {review_df['datetime'].min().year} to {review_df['datetime'].max().year}")
print(f"\n7. PRODUCT POPULARITY: Top 20 products account for a significant portion of reviews")

print("""

METADATA DATASET KEY INSIGHTS:
--------------------------------
""")

print(f"1. SCALE: {len(meta_df):,} unique products")
if len(price_data) > 0:
    print(f"\n2. PRICING: Median price ${price_data['price'].median():.2f}, mean ${price_data['price'].mean():.2f}")
    print(f"   - Free products: {len(free_products):,} ({len(free_products)/len(meta_df)*100:.1f}%)")
print(f"\n3. CONTENT RICHNESS:")
print(f"   - With features: {len(products_with_features)/len(meta_df)*100:.1f}%")
print(f"   - With descriptions: {len(products_with_desc)/len(meta_df)*100:.1f}%")
print(f"   - With images: {len(products_with_images)/len(meta_df)*100:.1f}%")
print(f"\n4. CATEGORIES: Main categories span both 'Software' and 'Appstore for Android'")

print("""

DATASET RELATIONSHIP INSIGHTS:
--------------------------------
""")

print(f"1. COVERAGE: {len(common_asins)/len(review_parent_asins)*100:.1f}% of reviewed products have metadata")
print(f"\n2. ORPHANED DATA:")
print(f"   - {len(reviews_only):,} products have reviews but no metadata")
print(f"   - {len(meta_only):,} products have metadata but no reviews")
if len(valid_comparison) > 0:
    print(f"\n3. RATING CONSISTENCY: Correlation between review avg and meta avg: {correlation.iloc[0, 1]:.3f}")

print("""

DATA QUALITY INSIGHTS:
-----------------------
""")

print(f"1. COMPLETENESS: Most review fields are well-populated (>99%)")
print(f"2. METADATA GAPS: Price ({len(meta_df[meta_df['price'].isna()])/len(meta_df)*100:.1f}% missing)")
print(f"3. DUPLICATE HANDLING: Minimal exact duplicates, but some users review same products multiple times")

print("""

RECOMMENDATIONS:
-----------------
1. For modeling: Focus on the {len(common_asins):,} products with both reviews and metadata
2. Handle missing data: Consider imputation strategies for price and ratings
3. Text analysis: Rich review text available for NLP tasks
4. Temporal patterns: Account for review recency in recommendations
5. User segmentation: Distinguish one-time vs. repeat reviewers
6. Rating normalization: Consider user and product biases given rating skew
7. Cold start: {len(meta_only):,} products need content-based recommendations
""")

## Summary

This notebook provided a comprehensive exploratory data analysis of the Amazon Software Reviews 2023 dataset, covering:

1. **Dataset Overview**: Structure, size, and basic statistics
2. **Missing Values**: Identification of gaps in both datasets
3. **Review Analysis**: Ratings, temporal patterns, text characteristics, user behavior, helpful votes, and products
4. **Metadata Analysis**: Categories, pricing, ratings, stores, features, descriptions, and media
5. **Relationships**: Dataset overlap, rating comparisons, and price-rating relationships
6. **Data Quality**: Duplicates, completeness, and quality issues
7. **Key Insights**: Actionable findings and recommendations

The analysis reveals a rich dataset suitable for recommender systems, with good coverage between reviews and metadata, though some data quality issues should be addressed during preprocessing.